# 01 - Continuous ISL Keypoint Extraction with MediaPipe Holistic

This notebook extracts **543 3D landmarks** (Pose: 33, Face: 468, Left Hand: 21, Right Hand: 21) from continuous ISL video files and saves cached `.npy` / `.npz` keypoint arrays for model training.

**Note:** By default, `MAX_SAMPLES = 5` is set to prevent accidental full-dataset processing when opened. Set `MAX_SAMPLES = None` for full batch extraction.

In [ ]:
# 1. Install & Verify Dependencies
!pip install -q mediapipe opencv-python tqdm numpy
import os, sys
from pathlib import Path

# Add training directory to sys.path
WORKING_DIR = Path("/kaggle/working/sentence-model-training") if os.path.exists("/kaggle") else Path("..")
sys.path.insert(0, str(WORKING_DIR.resolve()))

from config import KEYPOINTS_CACHE_DIR, VIDEO_DATA_DIR, MP_DETECTION_CONFIDENCE, MP_TRACKING_CONFIDENCE, MP_MODEL_COMPLEXITY
from src.preprocessing import MediaPipeHolisticExtractor
print("MediaPipe Holistic Extractor and Config imported successfully.")

In [ ]:
# 2. Configure Input & Output Paths
# Configurable paths (override as needed for ISLTranslate, iSign, etc.)
VIDEO_INPUT_DIR = Path("/kaggle/input/isltranslate/videos") if os.path.exists("/kaggle") else Path("./sample_videos")
KEYPOINT_OUTPUT_DIR = Path("/kaggle/working/keypoints_cache") if os.path.exists("/kaggle") else Path("./keypoints_cache")
SAVE_FORMAT = "npz"  # 'npz' or 'npy'

# Safety Limit: Process only a small batch by default for inspection
MAX_SAMPLES = 5  # Set to None to extract all videos

KEYPOINT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Reading videos from:    {VIDEO_INPUT_DIR}")
print(f"Saving keypoints to:     {KEYPOINT_OUTPUT_DIR}")
print(f"Max samples to extract:  {MAX_SAMPLES if MAX_SAMPLES is not None else 'ALL'}")

In [ ]:
# 3. Run Batch Keypoint Extraction with MediaPipe Holistic
import glob
import numpy as np
from tqdm import tqdm

extractor = MediaPipeHolisticExtractor(
    min_detection_confidence=MP_DETECTION_CONFIDENCE,
    min_tracking_confidence=MP_TRACKING_CONFIDENCE,
    model_complexity=MP_MODEL_COMPLEXITY,
)

video_files = list(VIDEO_INPUT_DIR.glob("*.mp4")) + list(VIDEO_INPUT_DIR.glob("*.avi"))
print(f"Found {len(video_files)} total videos in {VIDEO_INPUT_DIR}.")

if MAX_SAMPLES is not None:
    video_files = video_files[:MAX_SAMPLES]
    print(f"Extracting debug batch of {len(video_files)} videos...")

processed_count = 0
for vid in tqdm(video_files, desc="Extracting 543 Landmarks"):
    out_file = KEYPOINT_OUTPUT_DIR / f"{vid.stem}.{SAVE_FORMAT}"
    if out_file.exists():
        continue
    try:
        keypoints, frame_count = extractor.extract_video(vid)
        if SAVE_FORMAT == "npz":
            np.savez_compressed(out_file, keypoints=keypoints)
        else:
            np.save(out_file, keypoints)
        processed_count += 1
    except Exception as e:
        print(f"Error processing {vid.name}: {e}")

extractor.close()
print(f"Extraction complete! {processed_count} files processed and saved to {KEYPOINT_OUTPUT_DIR}.")